In [2]:
import requests
import pandas as pd
import json

In [3]:
app_id = "a388d69f"
app_key = "ba0adb7c5f9c130f8df1a6ac80c9fdaf"

base_url = "https://api.adzuna.com/v1/api/jobs/us/search/1"

In [4]:
params = {
    "app_id": app_id,
    "app_key": app_key,
    "what": "data analyst",
    "where": "texas",
    "distance": 50,
    "results_per_page": 50,
    "content-type": "application/json"
}

In [5]:
response = requests.get(base_url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [6]:
data = response.json()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['__CLASS__', 'count', 'results', 'mean'])


In [7]:
print("Total jobs found:", data['count'])
print("Jobs returned this page:", len(data['results']))
print("\nFirst job listing:")
print(json.dumps(data['results'][0], indent=2))

Total jobs found: 1154
Jobs returned this page: 50

First job listing:
{
  "latitude": 30.320233,
  "redirect_url": "https://www.adzuna.com/land/ad/5722245942?se=btq-a9Rb8RGUJNJexxfy6Q&utm_medium=api&utm_source=a388d69f&v=D8291760EA71689C87246BD2F5739437B308AC57",
  "location": {
    "area": [
      "US",
      "Texas",
      "Travis County",
      "Bee Cave"
    ],
    "__CLASS__": "Adzuna::API::Response::Location",
    "display_name": "Bee Cave, Travis County"
  },
  "salary_is_predicted": "1",
  "longitude": -97.865683,
  "company": {
    "display_name": "Apex Systems",
    "__CLASS__": "Adzuna::API::Response::Company"
  },
  "description": "Job: 3033642 Job Description: Data Analyst - Data Analyst I Location: Austin, Texas (Onsite) Employment Type: Contract Role Overview This position supports effective decision-making through economic, financial, and market analysis of critical issues. The role involves capacity planning, price and cost modeling, and evaluating returns on investme

In [8]:
jobs = []

for job in data['results']:
    jobs.append({
        "title": job.get("title"),
        "company": job["company"]["display_name"],
        "location": job["location"]["display_name"],
        "description": job.get("description"),
        "salary_min": job.get("salary_min"),
        "salary_max": job.get("salary_max"),
        "salary_predicted": job.get("salary_is_predicted"),
        "date_posted": job.get("created"),
        "category": job["category"]["label"]
    })

print("Jobs extracted:", len(jobs))
print("\nFirst job cleaned:")
print(jobs[0])

Jobs extracted: 50

First job cleaned:
{'title': 'Data Analyst - Data Analyst I', 'company': 'Apex Systems', 'location': 'Bee Cave, Travis County', 'description': 'Job: 3033642 Job Description: Data Analyst - Data Analyst I Location: Austin, Texas (Onsite) Employment Type: Contract Role Overview This position supports effective decision-making through economic, financial, and market analysis of critical issues. The role involves capacity planning, price and cost modeling, and evaluating returns on investment. The Data Analyst will also assist with capacity forecasting, budgeting, and reporting for operating expenses, capital, and resource allocation, whil…', 'salary_min': 63058.91, 'salary_max': 63058.91, 'salary_predicted': '1', 'date_posted': '2026-05-07T20:12:30Z', 'category': 'IT Jobs'}


In [9]:
import time

all_jobs = []
page = 1
total_pages = (data['count'] // 50) + 1

print(f"Total jobs: {data['count']}")
print(f"Total pages to pull: {total_pages}")

for page in range(1, total_pages + 1):
    url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print(f"Error on page {page}: {response.status_code}")
        break
    
    page_data = response.json()
    
    for job in page_data['results']:
        all_jobs.append({
            "title": job.get("title"),
            "company": job["company"]["display_name"],
            "location": job["location"]["display_name"],
            "description": job.get("description"),
            "salary_min": job.get("salary_min"),
            "salary_max": job.get("salary_max"),
            "salary_predicted": job.get("salary_is_predicted"),
            "date_posted": job.get("created"),
            "category": job["category"]["label"]
        })
    
    print(f"Page {page} pulled — {len(page_data['results'])} jobs")
    time.sleep(1)

print(f"\nTotal jobs collected: {len(all_jobs)}")

Total jobs: 1154
Total pages to pull: 24
Page 1 pulled — 50 jobs
Page 2 pulled — 50 jobs
Page 3 pulled — 50 jobs
Page 4 pulled — 50 jobs
Page 5 pulled — 50 jobs
Page 6 pulled — 50 jobs
Page 7 pulled — 50 jobs
Page 8 pulled — 50 jobs
Page 9 pulled — 50 jobs
Page 10 pulled — 50 jobs
Page 11 pulled — 50 jobs
Page 12 pulled — 50 jobs
Page 13 pulled — 50 jobs
Page 14 pulled — 50 jobs
Page 15 pulled — 50 jobs
Page 16 pulled — 50 jobs
Page 17 pulled — 50 jobs
Page 18 pulled — 50 jobs
Page 19 pulled — 50 jobs
Page 20 pulled — 50 jobs
Page 21 pulled — 50 jobs
Page 22 pulled — 50 jobs
Page 23 pulled — 50 jobs
Page 24 pulled — 3 jobs

Total jobs collected: 1153


In [10]:
df = pd.DataFrame(all_jobs)

print(df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 3 rows:")
df.head(3)

(1153, 9)

Column names:
['title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_predicted', 'date_posted', 'category']

First 3 rows:


,title,company,location,description,salary_min,salary_max,salary_predicted,date_posted,category
0,Data Analyst - Data Analyst I,Apex Systems,"Bee Cave, Travis County",Job: 3033642 Job Description: Data Analyst - D...,63058.91,63058.91,1,2026-05-07T20:12:30Z,IT Jobs
1,Sr. Data Analyst,HCLTech,"Texas, US","We are HCLTech, one of the fastest-growing lar...",169739.92,169739.92,1,2026-05-26T21:55:40Z,IT Jobs
2,Senior Data Analyst,HCLTech,"Texas, US","We are HCLTech, one of the fastest-growing lar...",166940.14,166940.14,1,2026-05-26T21:55:48Z,IT Jobs


In [11]:
# 1. Fix date format
df['date_posted'] = pd.to_datetime(df['date_posted']).dt.strftime('%B %d, %Y')

# 2. Create avg salary and drop min/max
df['avg_salary'] = (df['salary_min'] + df['salary_max']) / 2
df = df.drop(columns=['salary_min', 'salary_max'])

# 3. Replace salary_predicted 1/0 with Yes/No
df['salary_predicted'] = df['salary_predicted'].map({'1': 'Yes', '0': 'No'})

# 4. Extract city from location
df['city'] = df['location'].str.split(',').str[0]

# 5. Clean description - lowercase for skill matching later
df['description_clean'] = df['description'].str.lower()

print(df.shape)
df.head(3)

(1153, 10)


,title,company,location,description,salary_predicted,date_posted,category,avg_salary,city,description_clean
0,Data Analyst - Data Analyst I,Apex Systems,"Bee Cave, Travis County",Job: 3033642 Job Description: Data Analyst - D...,Yes,"May 07, 2026",IT Jobs,63058.91,Bee Cave,job: 3033642 job description: data analyst - d...
1,Sr. Data Analyst,HCLTech,"Texas, US","We are HCLTech, one of the fastest-growing lar...",Yes,"May 26, 2026",IT Jobs,169739.92,Texas,"we are hcltech, one of the fastest-growing lar..."
2,Senior Data Analyst,HCLTech,"Texas, US","We are HCLTech, one of the fastest-growing lar...",Yes,"May 26, 2026",IT Jobs,166940.14,Texas,"we are hcltech, one of the fastest-growing lar..."


In [12]:
skills = [
    "sql", "python", "power bi", "tableau", "excel",
    "r", "spark", "aws", "azure", "snowflake",
    "pandas", "matplotlib", "seaborn", "machine learning",
    "statistics", "dax", "etl", "looker", "databricks"
]

for skill in skills:
    df[skill] = df['description_clean'].str.contains(skill, na=False)

print("Skills extracted successfully")
df[['title'] + skills].head(3)

Skills extracted successfully


,title,sql,python,power bi,tableau,excel,r,spark,aws,azure,snowflake,pandas,matplotlib,seaborn,machine learning,statistics,dax,etl,looker,databricks
0,Data Analyst - Data Analyst I,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False
1,Sr. Data Analyst,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False
2,Senior Data Analyst,False,False,False,False,True,True,True,False,False,False,False,False,False,False,False,False,False,False,False


In [13]:
print(df['description_clean'][0])

job: 3033642 job description: data analyst - data analyst i location: austin, texas (onsite) employment type: contract role overview this position supports effective decision-making through economic, financial, and market analysis of critical issues. the role involves capacity planning, price and cost modeling, and evaluating returns on investment. the data analyst will also assist with capacity forecasting, budgeting, and reporting for operating expenses, capital, and resource allocation, whil…


In [14]:
skill_counts = df[skills].sum().sort_values(ascending=False)

print("Skill mentions across all job postings:")
print(skill_counts)
print(f"\nTotal jobs analyzed: {len(df)}")

Skill mentions across all job postings:
r                   1153
excel                 78
sql                   48
etl                   41
python                33
snowflake             28
azure                 25
power bi              24
machine learning      22
databricks            22
tableau               21
spark                 15
aws                   14
pandas                 1
statistics             1
matplotlib             0
seaborn                0
dax                    0
looker                 0
dtype: int64

Total jobs analyzed: 1153


In [15]:
import re

skills = [
    "sql", "python", "power bi", "tableau", "excel",
    "\\br\\b", "spark", "aws", "azure", "snowflake",
    "pandas", "matplotlib", "seaborn", "machine learning",
    "statistics", "dax", "etl", "looker", "databricks"
]

skill_names = [
    "sql", "python", "power bi", "tableau", "excel",
    "r", "spark", "aws", "azure", "snowflake",
    "pandas", "matplotlib", "seaborn", "machine learning",
    "statistics", "dax", "etl", "looker", "databricks"
]

for skill, name in zip(skills, skill_names):
    df[name] = df['description_clean'].str.contains(skill, na=False, regex=True)

skill_counts = df[skill_names].sum().sort_values(ascending=False)
print("Skill mentions across all job postings:")
print(skill_counts)

Skill mentions across all job postings:
excel               78
sql                 48
etl                 41
python              33
snowflake           28
azure               25
power bi            24
machine learning    22
databricks          22
tableau             21
spark               15
aws                 14
r                   12
pandas               1
statistics           1
matplotlib           0
seaborn              0
dax                  0
looker               0
dtype: int64


In [16]:
df.to_csv('da_jobs_texas.csv', index=False)

print("File saved successfully")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

File saved successfully
Shape: (1153, 29)
Columns: ['title', 'company', 'location', 'description', 'salary_predicted', 'date_posted', 'category', 'avg_salary', 'city', 'description_clean', 'sql', 'python', 'power bi', 'tableau', 'excel', 'r', 'spark', 'aws', 'azure', 'snowflake', 'pandas', 'matplotlib', 'seaborn', 'machine learning', 'statistics', 'dax', 'etl', 'looker', 'databricks']


In [18]:
import mysql.connector

connection = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root1234",
    database="job_market"
)

cursor = connection.cursor()

print("Connected to MySQL successfully")

Connected to MySQL successfully


In [19]:
import math

df_load = df[['title', 'company', 'location', 'city', 'description', 
              'description_clean', 'avg_salary', 'salary_predicted', 
              'date_posted', 'category', 'sql', 'python', 'power bi', 
              'tableau', 'excel', 'r', 'spark', 'aws', 'azure', 
              'snowflake', 'pandas', 'matplotlib', 'seaborn', 
              'machine learning', 'statistics', 'dax', 'etl', 
              'looker', 'databricks']]

insert_query = """
INSERT INTO da_jobs (
    title, company, location, city, description, description_clean,
    avg_salary, salary_predicted, date_posted, category,
    sql_skill, python_skill, power_bi, tableau, excel_skill,
    r_skill, spark, aws, azure, snowflake, pandas, matplotlib,
    seaborn, machine_learning, statistics, dax, etl, looker, databricks
) VALUES (
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
    %s, %s, %s, %s, %s, %s, %s, %s, %s
)
"""

rows_inserted = 0

for _, row in df_load.iterrows():
    values = tuple(
        None if (isinstance(v, float) and math.isnan(v)) else v 
        for v in row
    )
    cursor.execute(insert_query, values)
    rows_inserted += 1

connection.commit()
print(f"Successfully inserted {rows_inserted} rows into MySQL")

Successfully inserted 1153 rows into MySQL


In [20]:
# ============================================================
# SQL QUERIES FOR MYSQL WORKBENCH
# Run these in MySQL Workbench after loading the data
# ============================================================

# Query 1 — Skill mentions across all job postings
"""
SELECT 
    'SQL' as skill, SUM(sql_skill) as mentions FROM da_jobs
UNION ALL SELECT 'Python', SUM(python_skill) FROM da_jobs
UNION ALL SELECT 'Excel', SUM(excel_skill) FROM da_jobs
UNION ALL SELECT 'Power BI', SUM(power_bi) FROM da_jobs
UNION ALL SELECT 'Tableau', SUM(tableau) FROM da_jobs
UNION ALL SELECT 'ETL', SUM(etl) FROM da_jobs
UNION ALL SELECT 'Azure', SUM(azure) FROM da_jobs
UNION ALL SELECT 'Snowflake', SUM(snowflake) FROM da_jobs
UNION ALL SELECT 'Machine Learning', SUM(machine_learning) FROM da_jobs
UNION ALL SELECT 'Databricks', SUM(databricks) FROM da_jobs
UNION ALL SELECT 'Spark', SUM(spark) FROM da_jobs
UNION ALL SELECT 'AWS', SUM(aws) FROM da_jobs
UNION ALL SELECT 'R', SUM(r_skill) FROM da_jobs
ORDER BY mentions DESC;
"""

# Query 2 — Top cities by job count and avg salary
"""
SELECT city, COUNT(*) as job_count, ROUND(AVG(avg_salary), 0) as avg_salary
FROM da_jobs
WHERE city != 'Texas'
GROUP BY city
ORDER BY job_count DESC
LIMIT 10;
"""

# Query 3 — Skill percentage of total jobs
"""
SELECT 'Excel' as skill, SUM(excel_skill) as mentions,
    ROUND(SUM(excel_skill) * 100.0 / COUNT(*), 1) as pct_of_jobs FROM da_jobs
UNION ALL SELECT 'SQL', SUM(sql_skill), ROUND(SUM(sql_skill) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'Python', SUM(python_skill), ROUND(SUM(python_skill) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'ETL', SUM(etl), ROUND(SUM(etl) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'Power BI', SUM(power_bi), ROUND(SUM(power_bi) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'Tableau', SUM(tableau), ROUND(SUM(tableau) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'Azure', SUM(azure), ROUND(SUM(azure) * 100.0 / COUNT(*), 1) FROM da_jobs
UNION ALL SELECT 'Snowflake', SUM(snowflake), ROUND(SUM(snowflake) * 100.0 / COUNT(*), 1) FROM da_jobs
ORDER BY mentions DESC;
"""

# Query 4 — Most common job titles
"""
SELECT title, COUNT(*) as job_count
FROM da_jobs
GROUP BY title
ORDER BY job_count DESC
LIMIT 10;
"""

# Query 5 — Job postings by age
"""
SELECT 
    CASE 
        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 3 DAY)
        THEN 'Last 3 days'
        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 7 DAY)
        THEN 'Last 4-7 days'
        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)
        THEN 'Last 8-30 days'
        ELSE 'Older than 30 days'
    END as posting_age,
    COUNT(*) as job_count
FROM da_jobs
GROUP BY posting_age
ORDER BY job_count DESC;
"""

"\nSELECT \n    CASE \n        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 3 DAY)\n        THEN 'Last 3 days'\n        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 7 DAY)\n        THEN 'Last 4-7 days'\n        WHEN STR_TO_DATE(date_posted, '%M %d, %Y') >= DATE_SUB(CURDATE(), INTERVAL 30 DAY)\n        THEN 'Last 8-30 days'\n        ELSE 'Older than 30 days'\n    END as posting_age,\n    COUNT(*) as job_count\nFROM da_jobs\nGROUP BY posting_age\nORDER BY job_count DESC;\n"